# Prepare the Space Exploration Dataset

Run this notebook **once** before the other notebooks in `advanced-retrieval/`. It builds a single, relationship-rich corpus that every later notebook reuses, so you can ask the *same* questions with plain vector search, Microsoft Retriever 1, a topic graph and an entity graph and compare the results.

What it does:

1. **Fetches** about 230 curated English Wikipedia articles about space exploration and caches them locally
2. **Prepares Blob Storage**: turns on soft delete (required by the entity graph) and creates two containers
3. **Uploads** the full corpus to `space-full`, and a smaller, truncated subset to `space-entity` for the entity graph
4. **Builds an ingestion pipeline in Foundry IQ** (data source → skillset → index → indexer) that chunks and embeds the articles **server-side**, so nothing is embedded on your machine

![Dataset preparation: fetch Wikipedia articles, upload to two Blob containers, build a data source, indexer and skillset that chunk and embed server-side into the space-exploration index, used by the Retriever 1, topic graph and entity graph notebooks](../docs/images/prepare-space-dataset.svg)

## Why build the pipeline by hand?

A Blob **knowledge source** could generate the same data source, skillset, indexer and index for you in one call, but the generated objects follow a fixed template that you shouldn't edit. Here we want control over the index schema, because the **topic graph** notebook requires the chunk vector field to be **stored, retrievable, and use a cosine or dotProduct HNSW profile**. Building the objects explicitly also shows every step of the ingestion pipeline. (The entity graph notebook uses the knowledge-source shortcut, so you see both approaches.)

## Prerequisites

| Requirement | Why |
|---|---|
| Azure AI Search, **Basic tier or higher**, in a region that supports agentic retrieval, Microsoft Retriever 1 and knowledge graphs | Runs the indexer and hosts the index |
| Search service **system-assigned managed identity** with **Cognitive Services User** on the Foundry resource | The embedding skill and the query vectorizer call your embedding deployment without keys |
| Foundry resource with an embedding deployment (`text-embedding-3-large`) | Chunk and query embeddings |
| **Storage account**: general-purpose v2, **not** ADLS Gen2 (hierarchical namespace off). Shared-key access can stay **disabled**: everything here uses Microsoft Entra ID | Holds the corpus |
| Storage **Blob soft delete on**, **blob versioning off** | Required by the entity graph. Set both with the `az` command in Step 3 |
| Search service managed identity: **Storage Blob Data Reader** on the storage account | The indexer reads blobs with a `ResourceId=` connection, no keys |
| Your identity: **Storage Blob Data Contributor** on the storage account | Create containers and upload articles |
| Your identity: **Search Service Contributor** and **Search Index Data Contributor** (or an admin API key) | Create objects and read the index |
| `.env` with `AZURE_SEARCH_ENDPOINT`, `FOUNDRY_PROJECT_ENDPOINT`, `EMBEDDING_DEPLOYMENT`, `EMBEDDING_MODEL`, `AZURE_STORAGE_RESOURCE_ID` | See `.env.sample` |

> **Costs**: embedding roughly 5,000 chunks with `text-embedding-3-large` is inexpensive, but not free. The indexer runs on your search service.

>> **Licensing**: article text comes from English Wikipedia under [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). The cached text is git-ignored, and each chunk keeps its `source_url` for attribution.

## Step 1: Configuration

All notebooks in this repo use **`azure-search-documents==12.2.0b1`**, which targets the `2026-11-01-preview` REST API.

In [ ]:
# %pip install azure-search-documents==12.2.0b1 azure-identity azure-storage-blob python-dotenv requests tqdm

In [ ]:
import json
import os
import re
import time
import unicodedata
from pathlib import Path

import requests
from azure.core.credentials import AzureKeyCredential
from azure.core.exceptions import HttpResponseError
from azure.identity import DefaultAzureCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient, SearchIndexerClient
from azure.search.documents.indexes.models import (
    AzureOpenAIEmbeddingSkill,
    AzureOpenAIVectorizer,
    AzureOpenAIVectorizerParameters,
    HnswAlgorithmConfiguration,
    HnswParameters,
    IndexingParameters,
    IndexingParametersConfiguration,
    IndexProjectionMode,
    InputFieldMappingEntry,
    NativeBlobSoftDeleteDeletionDetectionPolicy,
    OutputFieldMappingEntry,
    SearchField,
    SearchFieldDataType,
    SearchIndex,
    SearchIndexer,
    SearchIndexerDataContainer,
    SearchIndexerDataSourceConnection,
    SearchIndexerIndexProjection,
    SearchIndexerIndexProjectionSelector,
    SearchIndexerIndexProjectionsParameters,
    SearchIndexerSkillset,
    SemanticConfiguration,
    SemanticField,
    SemanticPrioritizedFields,
    SemanticSearch,
    SplitSkill,
    VectorSearch,
    VectorSearchProfile,
)
from azure.search.documents.models import QueryType, VectorizableTextQuery
from azure.storage.blob import BlobServiceClient
from dotenv import load_dotenv
from tqdm import tqdm

load_dotenv(override=True)  # pick up .env edits without restarting the kernel

search_endpoint = os.environ["AZURE_SEARCH_ENDPOINT"]
search_api_key = os.environ.get("AZURE_SEARCH_API_KEY")
search_credential = AzureKeyCredential(search_api_key) if search_api_key else DefaultAzureCredential()

ai_services_endpoint = os.environ["FOUNDRY_PROJECT_ENDPOINT"].split("/api/projects")[0]
embedding_deployment = os.environ["EMBEDDING_DEPLOYMENT"]
embedding_model = os.environ["EMBEDDING_MODEL"]
EMBEDDING_DIMENSIONS = 3072  # text-embedding-3-large

# Keyless storage: the notebook uses your Entra identity, the search service uses its managed identity.
storage_resource_id = os.environ["AZURE_STORAGE_RESOURCE_ID"]
storage_account_name = storage_resource_id.rstrip("/").split("/")[-1]
storage_account_url = f"https://{storage_account_name}.blob.core.windows.net"
storage_search_connection = f"ResourceId={storage_resource_id};"

index_client = SearchIndexClient(search_endpoint, search_credential)
indexer_client = SearchIndexerClient(search_endpoint, search_credential)

print(f"Search endpoint:      {search_endpoint}")
print(f"AI Services endpoint: {ai_services_endpoint}")
print(f"Embedding:            {embedding_deployment} / {embedding_model}")
print(f"Storage account:      {storage_account_name}")
print(f"Auth:                 {'API key' if search_api_key else 'DefaultAzureCredential'}")

In [ ]:
DATA_DIR = Path("..") / "data" / "space-exploration"
CACHE_DIR = DATA_DIR / "cache"

CONTAINER_FULL = "space-full"
CONTAINER_ENTITY = "space-entity"
ENTITY_MAX_CHARS = 12_000  # caps LLM extraction cost per article in the entity graph

INDEX_NAME = "space-exploration"
DATA_SOURCE_NAME = "space-exploration-ds"
SKILLSET_NAME = "space-exploration-ss"
INDEXER_NAME = "space-exploration-ixr"


def read_titles(path: Path) -> list[str]:
    lines = path.read_text(encoding="utf-8").splitlines()
    return [line.strip() for line in lines if line.strip() and not line.startswith("#")]


all_titles = read_titles(DATA_DIR / "articles.txt")
entity_titles = read_titles(DATA_DIR / "entity-subset.txt")
fetch_titles = list(dict.fromkeys(all_titles + entity_titles))
print(f"{len(all_titles)} corpus titles, {len(entity_titles)} entity-subset titles, {len(fetch_titles)} unique to fetch")

## Step 2: Fetch the Wikipedia articles

Each article is fetched as plain text from the MediaWiki API and cached as JSON, so re-running the notebook only fetches what's missing. Requests are spaced out and automatically retried on `429 Too Many Requests` (honouring `Retry-After`).

Wikimedia's [User-Agent policy](https://meta.wikimedia.org/wiki/User-Agent_policy) asks clients to include contact details, and generic agents are throttled harder. If you still hit `429`, set `WIKI_USER_AGENT` in `.env`, for example `foundry-iq-space-demo/1.0 (you@example.com)`, and re-run; already cached articles are skipped.

In [ ]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

WIKI_API = "https://en.wikipedia.org/w/api.php"
# Wikimedia's User-Agent policy asks for contact info; generic agents are throttled harder.
WIKI_HEADERS = {"User-Agent": os.environ.get("WIKI_USER_AGENT", "foundry-iq-space-demo/1.0 (Azure AI Search sample notebooks)")}

wiki_session = requests.Session()
wiki_session.headers.update(WIKI_HEADERS)
wiki_session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=6, backoff_factor=2, status_forcelist=[429, 503], respect_retry_after_header=True,
)))


def slugify(title: str) -> str:
    ascii_title = unicodedata.normalize("NFKD", title).encode("ascii", "ignore").decode()
    return re.sub(r"[^A-Za-z0-9]+", "-", ascii_title).strip("-").lower()


def fetch_article(title: str) -> dict | None:
    params = {
        "action": "query", "format": "json", "formatversion": 2, "redirects": 1,
        "prop": "extracts|info", "explaintext": 1, "inprop": "url", "titles": title,
    }
    response = wiki_session.get(WIKI_API, params=params, timeout=30)
    response.raise_for_status()
    page = response.json()["query"]["pages"][0]
    if page.get("missing") or len(page.get("extract", "")) < 500:
        return None
    return {"title": page["title"], "url": page["fullurl"], "text": page["extract"]}

In [ ]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
articles, skipped = {}, []
progress = tqdm(fetch_titles, desc="Fetching articles", unit="article")
for title in progress:
    progress.set_postfix_str(title[:30])
    cache_file = CACHE_DIR / f"{slugify(title)}.json"
    if cache_file.exists():
        articles[title] = json.loads(cache_file.read_text(encoding="utf-8"))
        continue
    article = fetch_article(title)
    if article is None:
        skipped.append(title)
        continue
    cache_file.write_text(json.dumps(article, ensure_ascii=False), encoding="utf-8")
    articles[title] = article
    time.sleep(1)  # be polite to the Wikipedia API

print(f"Articles available: {len(articles)}")
if skipped:
    print(f"Skipped (missing or too short): {skipped}")

In [ ]:
total_chars = sum(len(a["text"]) for a in articles.values())
print(f"Total characters: {total_chars:,}  (~{total_chars // 1800:,} chunks at 2,000 chars with 200 overlap)")
sample = articles.get("Baikonur Cosmodrome") or next(iter(articles.values()))
print(f"\n{sample['title']}  ({sample['url']})\n{sample['text'][:600]}...")

## Step 3: Prepare Blob Storage

Everything here is **keyless**: the notebook connects with your Microsoft Entra identity, and the search service reads the blobs with its managed identity. Shared-key access can stay disabled on the account.

The entity graph (`entity-graph.ipynb`) needs two account-level settings **before its first indexing run**:

- **Native Blob soft delete: on.** Lets the indexer detect deleted blobs
- **Blob versioning: off**

Both are management-plane settings, so set them once with the Azure CLI (requires **Storage Account Contributor** or Owner on the account):

```bash
az storage account blob-service-properties update \
  --account-name <storage-account> --resource-group <resource-group> \
  --enable-delete-retention true --delete-retention-days 7 \
  --enable-versioning false
```

The next cell reports the soft-delete setting (if your role can read it) and creates the containers.

In [ ]:
blob_service = BlobServiceClient(storage_account_url, credential=DefaultAzureCredential())

try:
    retention = blob_service.get_service_properties()["delete_retention_policy"]
    print(f"Blob soft delete: {'on' if retention.enabled else 'OFF — run the az command above'}")
except HttpResponseError:
    print("Couldn't read blob service properties with your role; make sure you ran the az command above.")

for name in (CONTAINER_FULL, CONTAINER_ENTITY):
    container = blob_service.get_container_client(name)
    if not container.exists():
        container.create_container()
    print(f"Container ready: {name}")

## Step 4: Upload the articles

- `space-full` gets every article. The indexer reads custom blob metadata (`title`, `source_url`) as fields, so citations can point back to Wikipedia.
- `space-entity` gets the entity subset, **truncated to the first `ENTITY_MAX_CHARS` characters**. The lead sections of these articles carry the key facts (crew, launch site, agency), and capping length keeps entity extraction cost predictable.

Blob metadata must be ASCII, so titles are ASCII-folded for metadata only; the full title is also the first line of each text file.

In [ ]:
def upload_article(container_name: str, article: dict, max_chars: int | None = None) -> None:
    text = article["text"] if max_chars is None else article["text"][:max_chars]
    metadata = {
        "title": unicodedata.normalize("NFKD", article["title"]).encode("ascii", "ignore").decode(),
        "source_url": article["url"],
    }
    blob_service.get_blob_client(container_name, f"{slugify(article['title'])}.txt").upload_blob(
        f"{article['title']}\n\n{text}".encode("utf-8"), overwrite=True, metadata=metadata
    )

In [ ]:
corpus = [articles[t] for t in all_titles if t in articles]
entity_subset = [articles[t] for t in entity_titles if t in articles]

for article in tqdm(corpus, desc=f"Uploading to {CONTAINER_FULL}", unit="blob"):
    upload_article(CONTAINER_FULL, article)

for article in tqdm(entity_subset, desc=f"Uploading to {CONTAINER_ENTITY}", unit="blob"):
    upload_article(CONTAINER_ENTITY, article, max_chars=ENTITY_MAX_CHARS)

# Versioned accounts return a version_id on every blob.
versioned = any(b.version_id for b in blob_service.get_container_client(CONTAINER_ENTITY).list_blobs(include=["versions"]))
print("⚠️ Blob versioning is ON — disable it before running entity-graph.ipynb (see Step 3)." if versioned else "Blob versioning is off ✔")

## Step 5: Create the index

One search document per **chunk**. Key design choices:

| Field | Purpose |
|---|---|
| `chunk_id` | Key. Index projections generate it, which requires the `keyword` analyzer |
| `parent_id` | The source blob, so chunks can be grouped back into articles |
| `title`, `source_url` | From blob metadata; used for citations |
| `chunk` | Chunk text: keyword search, semantic ranking, and what retrieval returns |
| `text_vector` | 3,072-dimension embedding. **Stored and retrievable with a cosine HNSW profile**, which the topic graph build requires |

The **vectorizer** on the vector profile lets the service embed query text at query time, which knowledge bases and Microsoft Retriever 1 rely on.

In [ ]:
fields = [
    SearchField(name="chunk_id", type=SearchFieldDataType.String, key=True, searchable=True,
                analyzer_name="keyword", filterable=True, sortable=True),
    SearchField(name="parent_id", type=SearchFieldDataType.String, filterable=True),
    SearchField(name="title", type=SearchFieldDataType.String, searchable=True, filterable=True, facetable=True),
    SearchField(name="source_url", type=SearchFieldDataType.String),
    SearchField(name="chunk", type=SearchFieldDataType.String, searchable=True),
    SearchField(
        name="text_vector",
        type=SearchFieldDataType.Collection(SearchFieldDataType.Single),
        searchable=True, retrievable=True, stored=True,
        vector_search_dimensions=EMBEDDING_DIMENSIONS,
        vector_search_profile_name="hnsw-cosine-profile",
    ),
]

vector_search = VectorSearch(
    algorithms=[HnswAlgorithmConfiguration(name="hnsw-cosine", parameters=HnswParameters(metric="cosine"))],
    vectorizers=[AzureOpenAIVectorizer(
        vectorizer_name="aoai-vectorizer",
        parameters=AzureOpenAIVectorizerParameters(
            resource_url=ai_services_endpoint, deployment_name=embedding_deployment, model_name=embedding_model,
        ),
    )],
    profiles=[VectorSearchProfile(
        name="hnsw-cosine-profile", algorithm_configuration_name="hnsw-cosine", vectorizer_name="aoai-vectorizer",
    )],
)

semantic_search = SemanticSearch(
    default_configuration_name="semantic-config",
    configurations=[SemanticConfiguration(
        name="semantic-config",
        prioritized_fields=SemanticPrioritizedFields(
            title_field=SemanticField(field_name="title"),
            content_fields=[SemanticField(field_name="chunk")],
        ),
    )],
)

index = SearchIndex(
    name=INDEX_NAME,
    description="Chunks of English Wikipedia articles about space exploration: missions, astronauts, agencies, launch sites, spacecraft and telescopes.",
    fields=fields,
    vector_search=vector_search,
    semantic_search=semantic_search,
)
index_client.create_or_update_index(index)
print(f"Index '{INDEX_NAME}' created or updated")

## Step 6: Data source and skillset

**Data source:** points the indexer at the `space-full` container with a **managed-identity connection**: `ResourceId=/subscriptions/.../storageAccounts/<account>;`. The search service authenticates as its system-assigned identity, so it needs **Storage Blob Data Reader** on the storage account. No keys are stored.

**Skillset:** runs per article:

1. `SplitSkill` cuts `/document/content` into 2,000-character pages with 200 characters of overlap
2. `AzureOpenAIEmbeddingSkill` embeds each page. No API key is set, so it authenticates with the search service's managed identity
3. **Index projections** write each page to the index as its own document, copying the parent's `title` and `source_url`. `skipIndexingParentDocuments` means only chunks are stored

In [ ]:
data_source = SearchIndexerDataSourceConnection(
    name=DATA_SOURCE_NAME,
    type="azureblob",
    connection_string=storage_search_connection,
    container=SearchIndexerDataContainer(name=CONTAINER_FULL),
    data_deletion_detection_policy=NativeBlobSoftDeleteDeletionDetectionPolicy(),
)
indexer_client.create_or_update_data_source_connection(data_source)
print(f"Data source '{DATA_SOURCE_NAME}' created or updated")

In [ ]:
split_skill = SplitSkill(
    name="split-pages",
    context="/document",
    text_split_mode="pages",
    maximum_page_length=2000,
    page_overlap_length=200,
    inputs=[InputFieldMappingEntry(name="text", source="/document/content")],
    outputs=[OutputFieldMappingEntry(name="textItems", target_name="pages")],
)

embedding_skill = AzureOpenAIEmbeddingSkill(
    name="embed-pages",
    context="/document/pages/*",
    resource_url=ai_services_endpoint,
    deployment_name=embedding_deployment,
    model_name=embedding_model,
    dimensions=EMBEDDING_DIMENSIONS,
    inputs=[InputFieldMappingEntry(name="text", source="/document/pages/*")],
    outputs=[OutputFieldMappingEntry(name="embedding", target_name="text_vector")],
)

index_projection = SearchIndexerIndexProjection(
    selectors=[SearchIndexerIndexProjectionSelector(
        target_index_name=INDEX_NAME,
        parent_key_field_name="parent_id",
        source_context="/document/pages/*",
        mappings=[
            InputFieldMappingEntry(name="chunk", source="/document/pages/*"),
            InputFieldMappingEntry(name="text_vector", source="/document/pages/*/text_vector"),
            InputFieldMappingEntry(name="title", source="/document/title"),
            InputFieldMappingEntry(name="source_url", source="/document/source_url"),
        ],
    )],
    parameters=SearchIndexerIndexProjectionsParameters(projection_mode=IndexProjectionMode.SKIP_INDEXING_PARENT_DOCUMENTS),
)

skillset = SearchIndexerSkillset(
    name=SKILLSET_NAME,
    description="Chunk Wikipedia articles and embed each chunk.",
    skills=[split_skill, embedding_skill],
    index_projection=index_projection,
)
indexer_client.create_or_update_skillset(skillset)
print(f"Skillset '{SKILLSET_NAME}' created or updated")

## Step 7: Run the indexer

Creating the indexer starts it immediately. The next cell polls its status until the run finishes; expect a few minutes for about 230 articles.

In [ ]:
indexer = SearchIndexer(
    name=INDEXER_NAME,
    data_source_name=DATA_SOURCE_NAME,
    target_index_name=INDEX_NAME,
    skillset_name=SKILLSET_NAME,
    parameters=IndexingParameters(configuration=IndexingParametersConfiguration(
        data_to_extract="contentAndMetadata", parsing_mode="default",
    )),
)
indexer_client.create_or_update_indexer(indexer)
print(f"Indexer '{INDEXER_NAME}' created or updated")

In [ ]:
progress = tqdm(total=len(corpus), desc="Indexing articles", unit="article")
while True:
    last = indexer_client.get_indexer_status(INDEXER_NAME).last_result
    state = last.status if last else "notStarted"
    processed, failed = (last.item_count, last.failed_item_count) if last else (0, 0)
    progress.update(processed - progress.n)
    progress.set_postfix(state=state, failed=failed)
    if state not in ("inProgress", "notStarted", "reset"):
        break
    time.sleep(20)
progress.close()

for error in (last.errors or [])[:5]:
    print(f"  ERROR {error.key}: {error.error_message}")
for warning in (last.warnings or [])[:5]:
    print(f"  WARN  {warning.key}: {warning.message}")

## Step 8: Verify the index

Check the chunk count (the topic graph wants well over 1,000 chunks, since topic sampling draws from a candidate pool of 250–10,000), confirm the vector field settings the topic graph needs, and run a quick hybrid query.

In [ ]:
search_client = SearchClient(search_endpoint, INDEX_NAME, search_credential)
print(f"Chunks in '{INDEX_NAME}': {search_client.get_document_count():,}")

created_index = index_client.get_index(INDEX_NAME)
vector_field = next(f for f in created_index.fields if f.name == "text_vector")
metric = created_index.vector_search.algorithms[0].parameters.metric
print(f"text_vector: stored={vector_field.stored}, retrievable={vector_field.retrievable}, metric={metric}")

In [ ]:
question = "Which cosmodrome in Kazakhstan launched the first human into space?"
results = search_client.search(
    search_text=question,
    vector_queries=[VectorizableTextQuery(text=question, k_nearest_neighbors=5, fields="text_vector")],
    query_type=QueryType.SEMANTIC,
    select=["title", "source_url", "chunk"],
    top=3,
)
for doc in results:
    print(f"\n[{doc['title']}] {doc['source_url']}\n  {doc['chunk'][:200]}...")

## Summary

You now have:

| Resource | Name | Used by |
|---|---|---|
| Blob container (full corpus) | `space-full` | This notebook's indexer |
| Blob container (entity subset) | `space-entity` | `entity-graph.ipynb` |
| Index | `space-exploration` | `microsoft-retriever-1.ipynb`, `topic-graph.ipynb` |
| Data source, skillset, indexer | `space-exploration-ds`, `-ss`, `-ixr` | Re-run the indexer if you add articles |

### Next steps

- [microsoft-retriever-1.ipynb](microsoft-retriever-1.ipynb): agentic search and knowledge bases with Microsoft Retriever 1
- [topic-graph.ipynb](topic-graph.ipynb): broad, theme-level questions with a topic graph
- [entity-graph.ipynb](entity-graph.ipynb): multi-hop relationship questions with an entity graph

### Clean up (optional)

Only run this when you've finished **all** notebooks in `02-*` and `03-*`, since they depend on these resources.

In [ ]:
# indexer_client.delete_indexer(INDEXER_NAME)
# indexer_client.delete_skillset(SKILLSET_NAME)
# indexer_client.delete_data_source_connection(DATA_SOURCE_NAME)
# index_client.delete_index(INDEX_NAME)
# for name in (CONTAINER_FULL, CONTAINER_ENTITY):
#     blob_service.delete_container(name)
# print("Deleted indexer, skillset, data source, index and containers")